# Giant Comparison Table

Six rows × four scenarios × five metrics, aggregated over 5 seeds.

**Methods (rows):**
1. **SF** — Social Force baseline (`trained_models/SF`)
2. **ORCA** — ORCA baseline (`trained_models/ORCA`)
3. **CrowdNav++** — GST predictor baseline (`trained_models/GST_predictor_rand`)
4. **GenSafeNav (conservative)** — LoraF rank 1 with LoRA branch **off** (`always_off`)
5. **GenSafeNav (aggressive)** — Full fine-tune on the invi→visi setup (`FullFineTune_invi_visi`, `always_off`)
6. **Ours (adaptive GT)** — LoraF rank 1 with `adaptive_gt` behaviour

**Scenarios (columns):** seperate_all_aware, seperate_all_ignorant, seperate_mixed_5050, cluster_aware_ignorant.

**Metrics:** Success Rate (SR), Collision Rate (CR), Nav Time (NT), Path Length (PL), Intrusion Time Ratio (ITR). SR/CR/ITR shown as percent. Each cell is `mean ± sd` across 5 seeds (42, 1000, 2000, 3000, 4000).

In [1]:
import os, json
from collections import defaultdict
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# ----------------------- config -----------------------
SCENARIOS = [
    'seperate_all_aware',
    'seperate_all_ignorant',
    'seperate_mixed_5050',
    'cluster_aware_ignorant',
]

# Baseline seeds use the EXP_ID_seed=N_N pattern from the patched
# test_baselines.sh. LoRA-based rows use plain seed labels.
SEEDS_BASELINE = {'1000_1000', '2000_2000', '3000_3000', '4000_4000'}
SEEDS_LORA     = {'42', '1000', '2000', '3000', '4000'}

# (display_name, source_dir, behaviour_suffix, seed_set)
METHODS = [
    ('SF',                       'trained_models/SF',                       'always_off',  SEEDS_BASELINE),
    ('ORCA',                     'trained_models/ORCA',                     'always_off',  SEEDS_BASELINE),
    ('CrowdNav++',               'trained_models/GST_predictor_rand',       'always_off',  SEEDS_BASELINE),
    ('GenSafeNav (conservative)','trained_models/LoraF_invi_visi_rank_1',   'always_off',  SEEDS_LORA),
    ('GenSafeNav (aggressive)',  'trained_models/FullFineTune_invi_visi',   'always_off',  SEEDS_BASELINE),
    ('Ours (adaptive GT)',       'trained_models/LoraF_invi_visi_rank_1',   'adaptive_gt', SEEDS_LORA),
]

# metric key in summary → (display name, in_percent, decimals)
METRICS = [
    ('success_rate',           'SR (%)',   True,  2),
    ('collision_rate',         'CR (%)',   True,  2),
    ('avg_nav_time',           'NT (s)',   False, 2),
    ('avg_path_length',        'PL (m)',   False, 2),
    ('avg_intrusion_ratio_pct','ITR (%)',  False, 2),  # already in percent
    ('avg_min_social_distance','SocD (m)', False, 3),  # mean per-episode min human–robot distance
]

In [2]:
# ----------------------- load -----------------------
def load_method(model_dir, behaviour, seed_set):
    """Return {scenario: {seed: {metric: value}}}."""
    path = os.path.join(model_dir, 'test', 'all_evaluations.json')
    out = {sc: {} for sc in SCENARIOS}
    if not os.path.exists(path):
        return out
    d = json.load(open(path))
    for k, v in d.items():
        if '_exp' not in k:
            continue
        base, exp = k.rsplit('_exp', 1)
        if exp not in seed_set:
            continue
        if not base.endswith('_' + behaviour):
            continue
        sc = base[:-(len(behaviour)+1)]
        if sc not in SCENARIOS:
            continue
        out[sc][exp] = v.get('summary', {})
    return out

loaded = {name: load_method(d, beh, seeds) for name, d, beh, seeds in METHODS}

# Coverage report
print(f"{'method':<28} {'scenario':<26} {'expected':<5}  seeds_found")
for name, _, _, seeds in METHODS:
    for sc in SCENARIOS:
        found = sorted(loaded[name][sc].keys())
        marker = '✓' if seeds.issubset(found) else 'PARTIAL' if found else 'EMPTY'
        print(f"  {name:<26} {sc:<26} {len(seeds):<5}  {marker:<8} {found}")

method                       scenario                   expected  seeds_found
  SF                         seperate_all_aware         4      ✓        ['1000_1000', '2000_2000', '3000_3000', '4000_4000']
  SF                         seperate_all_ignorant      4      ✓        ['1000_1000', '2000_2000', '3000_3000', '4000_4000']
  SF                         seperate_mixed_5050        4      ✓        ['1000_1000', '2000_2000', '3000_3000', '4000_4000']
  SF                         cluster_aware_ignorant     4      ✓        ['1000_1000', '2000_2000', '3000_3000', '4000_4000']
  ORCA                       seperate_all_aware         4      ✓        ['1000_1000', '2000_2000', '3000_3000', '4000_4000']
  ORCA                       seperate_all_ignorant      4      ✓        ['1000_1000', '2000_2000', '3000_3000', '4000_4000']
  ORCA                       seperate_mixed_5050        4      ✓        ['1000_1000', '2000_2000', '3000_3000', '4000_4000']
  ORCA                       cluster_aware_igno

In [3]:
# ----------------------- aggregate → mean ± sd -----------------------
def fmt(mean, sd, in_pct, decimals):
    if mean is None or (isinstance(mean, float) and np.isnan(mean)):
        return '—'
    if in_pct:
        mean, sd = mean * 100, sd * 100
    return f"{mean:.{decimals}f} ± {sd:.{decimals}f}"

rows = []
for name, _, _, _ in METHODS:
    for sc in SCENARIOS:
        seed_dicts = loaded[name][sc]
        row = {'Method': name, 'Scenario': sc}
        for mkey, mlabel, in_pct, dec in METRICS:
            vals = [s.get(mkey) for s in seed_dicts.values() if s.get(mkey) is not None]
            if vals:
                m = float(np.mean(vals))
                sd = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
                row[mlabel] = fmt(m, sd, in_pct, dec)
            else:
                row[mlabel] = '—'
        rows.append(row)

df = pd.DataFrame(rows)
df

,Method,Scenario,SR (%),CR (%),NT (s),PL (m),ITR (%),SocD (m)
0,SF,seperate_all_aware,18.80 ± 2.40,0.20 ± 0.23,28.80 ± 0.97,40.90 ± 0.76,1.52 ± 0.14,0.437 ± 0.022
1,SF,seperate_all_ignorant,13.60 ± 2.47,20.40 ± 3.88,30.05 ± 1.01,35.15 ± 0.67,3.44 ± 0.38,0.429 ± 0.007
2,SF,seperate_mixed_5050,15.60 ± 2.99,11.20 ± 2.99,28.39 ± 1.64,38.06 ± 1.05,2.65 ± 0.58,0.429 ± 0.012
3,SF,cluster_aware_ignorant,13.80 ± 1.74,0.60 ± 0.52,28.53 ± 3.49,39.74 ± 0.31,0.34 ± 0.09,0.418 ± 0.029
4,ORCA,seperate_all_aware,93.10 ± 1.10,0.70 ± 0.20,23.32 ± 0.72,22.98 ± 0.49,0.54 ± 0.08,0.478 ± 0.017
5,ORCA,seperate_all_ignorant,66.40 ± 3.88,29.20 ± 2.71,23.68 ± 0.82,20.01 ± 0.40,1.04 ± 0.16,0.492 ± 0.009
6,ORCA,seperate_mixed_5050,83.70 ± 1.91,11.80 ± 1.48,23.87 ± 0.41,21.98 ± 0.32,0.98 ± 0.14,0.488 ± 0.013
7,ORCA,cluster_aware_ignorant,89.60 ± 2.24,6.90 ± 1.89,18.94 ± 0.51,21.29 ± 0.48,0.59 ± 0.17,0.472 ± 0.023
8,CrowdNav++,seperate_all_aware,99.60 ± 0.46,0.40 ± 0.46,11.97 ± 0.17,19.00 ± 0.10,7.29 ± 0.50,0.417 ± 0.011
9,CrowdNav++,seperate_all_ignorant,89.20 ± 2.36,10.80 ± 2.36,13.73 ± 0.26,19.99 ± 0.26,7.96 ± 0.46,0.416 ± 0.006


In [4]:
# ----------------------- compact pivoted view: rows=Method, columns=(Scenario, Metric) -----------------------
long = df.melt(id_vars=['Method', 'Scenario'], var_name='Metric', value_name='Value')
wide = long.pivot_table(index='Method', columns=['Scenario', 'Metric'], values='Value', aggfunc='first')
wide = wide.reindex([m[0] for m in METHODS])
metric_order = [m[1] for m in METRICS]
wide = wide.reindex(columns=pd.MultiIndex.from_product([SCENARIOS, metric_order], names=['Scenario', 'Metric']))
wide

Scenario                  seperate_all_aware                                                                      seperate_all_ignorant                                                          \
Metric                                SR (%)       CR (%)        NT (s)        PL (m)      ITR (%)       SocD (m)                SR (%)        CR (%)        NT (s)        PL (m)       ITR (%)   
Method                                                                                                                                                                                            
SF                              18.80 ± 2.40  0.20 ± 0.23  28.80 ± 0.97  40.90 ± 0.76  1.52 ± 0.14  0.437 ± 0.022          13.60 ± 2.47  20.40 ± 3.88  30.05 ± 1.01  35.15 ± 0.67   3.44 ± 0.38   
ORCA                            93.10 ± 1.10  0.70 ± 0.20  23.32 ± 0.72  22.98 ± 0.49  0.54 ± 0.08  0.478 ± 0.017          66.40 ± 3.88  29.20 ± 2.71  23.68 ± 0.82  20.01 ± 0.40   1.04 ± 0.16   
CrowdNav++                      99.60 ± 0.46  0.40 ± 0.46  11.97 ± 0.17  19.00 ± 0.10  7.29 ± 0.50  0.417 ± 0.011          89.20 ± 2.36  10.80 ± 2.36  13.73 ± 0.26  19.99 ± 0.26   7.96 ± 0.46   
GenSafeNav (conservative)      100.00 ± 0.00  0.00 ± 0.00  16.24 ± 0.47  23.49 ± 0.64  2.61 ± 0.32  0.439 ± 0.011          97.20 ± 0.40   2.80 ± 0.40  16.59 ± 0.47  23.53 ± 0.62   2.30 ± 0.29   
GenSafeNav (aggressive)         99.30 ± 0.38  0.70 ± 0.38  10.69 ± 0.07  17.88 ± 0.16  8.28 ± 0.85  0.426 ± 0.008          59.50 ± 2.47  40.50 ± 2.47  11.61 ± 0.20  16.88 ± 0.29  12.62 ± 0.33   
Ours (adaptive GT)              99.28 ± 0.33  0.72 ± 0.33  10.94 ± 0.11  18.16 ± 0.28  7.33 ± 0.74  0.417 ± 0.011          97.04 ± 0.36   2.96 ± 0.36  16.51 ± 0.44  23.44 ± 0.59   2.39 ± 0.25   

Scenario                                 seperate_mixed_5050                                                                        cluster_aware_ignorant                                            \
Metric                          SocD (m)              SR (%)        CR (%)        NT (s)        PL (m)       ITR (%)       SocD (m)                 SR (%)        CR (%)        NT (s)        PL (m)   
Method                                                                                                                                                                                                 
SF                         0.429 ± 0.007        15.60 ± 2.99  11.20 ± 2.99  28.39 ± 1.64  38.06 ± 1.05   2.65 ± 0.58  0.429 ± 0.012           13.80 ± 1.74   0.60 ± 0.52  28.53 ± 3.49  39.74 ± 0.31   
ORCA                       0.492 ± 0.009        83.70 ± 1.91  11.80 ± 1.48  23.87 ± 0.41  21.98 ± 0.32   0.98 ± 0.14  0.488 ± 0.013           89.60 ± 2.24   6.90 ± 1.89  18.94 ± 0.51  21.29 ± 0.48   
CrowdNav++                 0.416 ± 0.006        95.40 ± 1.24   4.60 ± 1.24  13.06 ± 0.20  19.75 ± 0.23   7.36 ± 0.73  0.421 ± 0.005           95.60 ± 1.57   4.40 ± 1.57  12.68 ± 0.11  19.42 ± 0.21   
GenSafeNav (conservative)  0.451 ± 0.004        98.56 ± 0.73   1.36 ± 0.73  16.61 ± 0.58  23.79 ± 0.74   2.62 ± 0.23  0.439 ± 0.010           97.12 ± 0.77   2.88 ± 0.77  15.47 ± 0.68  22.40 ± 0.89   
GenSafeNav (aggressive)    0.421 ± 0.006        82.40 ± 4.79  17.60 ± 4.79  11.36 ± 0.20  17.82 ± 0.41  10.80 ± 0.35  0.421 ± 0.009           82.90 ± 2.43  17.10 ± 2.43  10.83 ± 0.10  17.43 ± 0.33   
Ours (adaptive GT)         0.448 ± 0.005        95.52 ± 1.53   4.48 ± 1.53  13.63 ± 0.40  20.64 ± 0.53   5.87 ± 0.39  0.431 ± 0.006           95.36 ± 0.92   4.64 ± 0.92  12.85 ± 0.30  19.83 ± 0.58   

Scenario                                               
Metric                         ITR (%)       SocD (m)  
Method                                                 
SF                         0.34 ± 0.09  0.418 ± 0.029  
ORCA                       0.59 ± 0.17  0.472 ± 0.023  
CrowdNav++                 5.71 ± 0.16  0.425 ± 0.002  
GenSafeNav (conservative)  3.89 ± 0.40  0.431 ± 0.002  
GenSafeNav (aggressive)    8.78 ± 0.46  0.421 ± 0.0

In [5]:
# ============================================================
# Table A — SD ACROSS SCENARIOS (per method × metric)
#   cell = std over the 4 scenario means (each scenario mean is itself
#   the mean over its seed-set). Big number → method is sensitive to scenario.
# ============================================================
def per_method_scenario_seed_values(name, scenario, mkey):
    return [s.get(mkey) for s in loaded[name][scenario].values() if s.get(mkey) is not None]

rows_a = []
for name, _, _, _ in METHODS:
    row = {'Method': name}
    for mkey, mlabel, in_pct, dec in METRICS:
        scen_means = []
        for sc in SCENARIOS:
            vals = per_method_scenario_seed_values(name, sc, mkey)
            if vals:
                scen_means.append(float(np.mean(vals)))
        if len(scen_means) > 1:
            sd = float(np.std(scen_means, ddof=1))
            if in_pct: sd *= 100
            row[mlabel] = round(sd, dec)
        else:
            row[mlabel] = np.nan
    rows_a.append(row)

df_sd_scenarios = pd.DataFrame(rows_a).set_index('Method')
df_sd_scenarios

,SR (%),CR (%),NT (s),PL (m),ITR (%),SocD (m)
Method,,,,,,
SF,2.41,9.65,0.76,2.50,1.35,0.008
ORCA,11.85,12.24,2.35,1.25,0.26,0.009
CrowdNav++,4.29,4.29,0.74,0.43,0.96,0.004
GenSafeNav (conservative),1.36,1.37,0.53,0.61,0.70,0.008
GenSafeNav (aggressive),16.36,16.36,0.44,0.46,1.99,0.003
Ours (adaptive GT),1.82,1.82,2.31,2.20,2.16,0.014


In [6]:
# ============================================================
# Table B — SD ACROSS EVAL SEEDS (per method × scenario × metric)
#   cell = std over the seeds, kept SEPARATE per scenario (not averaged).
#   Rows = Method; Columns = MultiIndex(Scenario, Metric).
# ============================================================
rows_b = []
for name, _, _, _ in METHODS:
    row = {'Method': name}
    for sc in SCENARIOS:
        for mkey, mlabel, in_pct, dec in METRICS:
            vals = per_method_scenario_seed_values(name, sc, mkey)
            if len(vals) > 1:
                sd = float(np.std(vals, ddof=1))
                if in_pct: sd *= 100
                row[(sc, mlabel)] = round(sd, dec)
            else:
                row[(sc, mlabel)] = np.nan
    rows_b.append(row)

df_sd_seeds = pd.DataFrame(rows_b).set_index('Method')
df_sd_seeds.columns = pd.MultiIndex.from_tuples(df_sd_seeds.columns, names=['Scenario', 'Metric'])
df_sd_seeds = df_sd_seeds.reindex(columns=pd.MultiIndex.from_product(
    [SCENARIOS, [m[1] for m in METRICS]], names=['Scenario', 'Metric']))
df_sd_seeds

Scenario                  seperate_all_aware                                       seperate_all_ignorant                                       seperate_mixed_5050                               \
Metric                                SR (%) CR (%) NT (s) PL (m) ITR (%) SocD (m)                SR (%) CR (%) NT (s) PL (m) ITR (%) SocD (m)              SR (%) CR (%) NT (s) PL (m) ITR (%)   
Method                                                                                                                                                                                            
SF                                      2.40   0.23   0.97   0.76    0.14    0.022                  2.47   3.88   1.01   0.67    0.38    0.007                2.99   2.99   1.64   1.05    0.58   
ORCA                                    1.10   0.20   0.72   0.49    0.08    0.017                  3.88   2.71   0.82   0.40    0.16    0.009                1.91   1.48   0.41   0.32    0.14   
CrowdNav++                              0.46   0.46   0.17   0.10    0.50    0.011                  2.36   2.36   0.26   0.26    0.46    0.006                1.24   1.24   0.20   0.23    0.73   
GenSafeNav (conservative)               0.00   0.00   0.47   0.64    0.32    0.011                  0.40   0.40   0.47   0.62    0.29    0.004                0.73   0.73   0.58   0.74    0.23   
GenSafeNav (aggressive)                 0.38   0.38   0.07   0.16    0.85    0.008                  2.47   2.47   0.20   0.29    0.33    0.006                4.79   4.79   0.20   0.41    0.35   
Ours (adaptive GT)                      0.33   0.33   0.11   0.28    0.74    0.011                  0.36   0.36   0.44   0.59    0.25    0.005                1.53   1.53   0.40   0.53    0.39   

Scenario                           cluster_aware_ignorant                                        
Metric                    SocD (m)                 SR (%) CR (%) NT (s) PL (m) ITR (%) SocD (m)  
Method                                                                                           
SF                           0.012                   1.74   0.52   3.49   0.31    0.09    0.029  
ORCA                         0.013                   2.24   1.89   0.51   0.48    0.17    0.023  
CrowdNav++                   0.005                   1.57   1.57   0.11   0.21    0.16    0.002  
GenSafeNav (conservative)    0.010                   0.77   0.77   0.68   0.89    0.40    0.002  
GenSafeNav (aggressive)      0.009                   2.43   2.43   0.10   0.33    0.46    0.007  
Ours (adaptive GT)           0.006                   0.92   0.92   0.30   0.58    0.75    0.007

In [7]:
# ============================================================
# Table C — SD ACROSS EPISODES (single seed)
#   Within one seed's test run (~250–1250 episodes), what's the std
#   over the episodes themselves?
#   - SR / CR: Bernoulli sample std = sqrt(p*(1-p))   (per-episode binary)
#   - NT     : std_nav_time             (already in summary)
#   - PL     : std_path_length          (already in summary)
#   - ITR    : std_intrusion_ratio_pct  (already in summary; already pct)
#   - SocD   : std_min_social_distance  (already in summary)
# Uses the lexicographically-smallest available seed per method.
# ============================================================

SD_KEY_FOR_MEAN = {
    'success_rate':            None,
    'collision_rate':          None,
    'avg_nav_time':            'std_nav_time',
    'avg_path_length':         'std_path_length',
    'avg_intrusion_ratio_pct': 'std_intrusion_ratio_pct',
    'avg_min_social_distance': 'std_min_social_distance',
}

rows_c = []
for name, _, _, seeds in METHODS:
    row = {'Method': name}
    target_seed = sorted(seeds)[0]
    for sc in SCENARIOS:
        sums = loaded[name][sc].get(target_seed, {})
        for mkey, mlabel, in_pct, dec in METRICS:
            sd_key = SD_KEY_FOR_MEAN[mkey]
            if sd_key is None:
                p = sums.get(mkey)
                sd = (p * (1 - p)) ** 0.5 if p is not None else None
            else:
                sd = sums.get(sd_key)
            if sd is None:
                row[(sc, mlabel)] = np.nan
            else:
                v = sd * 100 if in_pct else sd
                row[(sc, mlabel)] = round(float(v), dec)
    rows_c.append(row)

df_sd_episodes = pd.DataFrame(rows_c).set_index('Method')
df_sd_episodes.columns = pd.MultiIndex.from_tuples(df_sd_episodes.columns, names=['Scenario', 'Metric'])
df_sd_episodes = df_sd_episodes.reindex(columns=pd.MultiIndex.from_product(
    [SCENARIOS, [m[1] for m in METRICS]], names=['Scenario', 'Metric']))
df_sd_episodes

Scenario                  seperate_all_aware                                       seperate_all_ignorant                                       seperate_mixed_5050                               \
Metric                                SR (%) CR (%) NT (s) PL (m) ITR (%) SocD (m)                SR (%) CR (%) NT (s) PL (m) ITR (%) SocD (m)              SR (%) CR (%) NT (s) PL (m) ITR (%)   
Method                                                                                                                                                                                            
SF                                     40.59   6.31  12.21   8.71    2.98    0.130                 37.74  36.66  12.98  12.64    6.91    0.127               39.70  31.54  11.38  11.79    6.26   
ORCA                                   22.99   8.91   9.53   6.74    1.32    0.090                 46.17  44.50  10.60   7.49    3.41    0.093               38.42  34.28  10.42   7.58    2.80   
CrowdNav++                              8.91   8.91   3.40   4.65    8.91    0.138                 28.90  28.90   4.62   5.88    9.85    0.142               20.51  20.51   4.04   5.08    8.78   
GenSafeNav (conservative)               0.00   0.00   5.02   5.82    4.42    0.114                 16.50  16.50   5.17   6.08    3.77    0.109               12.55  12.55   5.20   6.20    5.04   
GenSafeNav (aggressive)                 6.31   6.31   2.31   4.16   10.24    0.141                 49.42  49.42   2.74   4.46   12.21    0.132               37.39  37.39   3.05   4.72   11.41   
Ours (adaptive GT)                      8.91   8.91   2.32   4.02    9.81    0.134                 16.50  16.50   5.45   6.30    4.10    0.114               17.60  17.60   3.66   4.94    8.36   

Scenario                           cluster_aware_ignorant                                        
Metric                    SocD (m)                 SR (%) CR (%) NT (s) PL (m) ITR (%) SocD (m)  
Method                                                                                           
SF                           0.141                  33.85   8.91  11.60   7.88    1.19    0.152  
ORCA                         0.102                  29.46  24.48   9.37   6.70    4.12    0.147  
CrowdNav++                   0.136                  22.20  22.20   3.63   5.07    9.58    0.143  
GenSafeNav (conservative)    0.128                  19.60  19.60   5.18   6.63   10.62    0.128  
GenSafeNav (aggressive)      0.137                  38.42  38.42   2.45   4.38   11.68    0.135  
Ours (adaptive GT)           0.139                  22.20  22.20   3.68   5.43   12.51    0.132

In [8]:
# ----------------------- save outputs -----------------------
df.to_csv('giant_table_long.csv', index=False)
wide.to_csv('giant_table_wide.csv')
df_sd_scenarios.to_csv('giant_table_sd_across_scenarios.csv')
df_sd_seeds.to_csv('giant_table_sd_across_seeds.csv')
df_sd_episodes.to_csv('giant_table_sd_across_episodes_seed42.csv')

try:
    latex = wide.to_latex(na_rep='—', multicolumn=True, escape=True)
    with open('giant_table.tex', 'w') as f:
        f.write(latex)
    print('Wrote:')
    print('  giant_table_long.csv, giant_table_wide.csv, giant_table.tex')
    print('  giant_table_sd_across_scenarios.csv')
    print('  giant_table_sd_across_seeds.csv')
    print('  giant_table_sd_across_episodes_seed42.csv')
except Exception as e:
    print(f'CSVs written. LaTeX export failed: {e}')

Wrote:
  giant_table_long.csv, giant_table_wide.csv, giant_table.tex
  giant_table_sd_across_scenarios.csv
  giant_table_sd_across_seeds.csv
  giant_table_sd_across_episodes_seed42.csv


# LoRA Scale Ablation

Single source: `trained_models/LoraF_invi_visi_rank_1`. Rows are the LoRA branch's `dynamic_scale` setting:

- **scale 0.0** — LoRA branch disabled (= `always_off` = `GenSafeNav (conservative)` row above).
- **scale 0.2 / 0.4 / 0.6 / 0.8 / 1.0** — LoRA branch held at a fixed value (`--lora_behaviour fixed_scale --lora_scale X`). Test runs use `--exp_id scale_X_<seed>` so keys look like `<scenario>_fixed_scale_expscale_X_<seed>`.
- **adaptive_gt** — dynamic scale interpolated from ground-truth awareness signal (= `Ours (adaptive GT)` row above).

Each cell = `mean ± seed-SD` (SR/CR/ITR in %, NT/PL/SocD in physical units). Empty cells (`—`) mean the test for that (scale, scenario, seed) hasn't been run yet.

In [9]:
# ----------------------- LoRA ablation config -----------------------
ABL_DIR   = 'trained_models/LoraF_invi_visi_rank_1'
ABL_SEEDS = ['42', '1000', '2000', '3000', '4000']

def _scale_seeds(scale_str):
    # test_lora_ablation.sh writes --exp_id "scale_<S>_<seed>"
    return {f'scale_{scale_str}_{s}' for s in ABL_SEEDS}

# (row_label, behaviour, seed_set)
ABL_METHODS = [
    ('scale 0.0 (off)',  'always_off',  set(ABL_SEEDS)),
    ('scale 0.2',        'fixed_scale', _scale_seeds('0.2')),
    ('scale 0.4',        'fixed_scale', _scale_seeds('0.4')),
    ('scale 0.6',        'fixed_scale', _scale_seeds('0.6')),
    ('scale 0.8',        'fixed_scale', _scale_seeds('0.8')),
    ('scale 1.0',        'fixed_scale', _scale_seeds('1.0')),
    ('adaptive_gt',      'adaptive_gt', set(ABL_SEEDS)),
]

abl_loaded = {label: load_method(ABL_DIR, beh, seeds) for label, beh, seeds in ABL_METHODS}

# Coverage report
print(f"{'row':<22} {'scenario':<26} {'expect':<6} marker  found")
for label, _, seeds in ABL_METHODS:
    for sc in SCENARIOS:
        found = sorted(abl_loaded[label][sc].keys())
        marker = '✓' if seeds.issubset(found) else 'PARTIAL' if found else 'EMPTY'
        print(f"  {label:<20} {sc:<26} {len(seeds):<6} {marker:<7} {found}")

row                    scenario                   expect marker  found
  scale 0.0 (off)      seperate_all_aware         5      ✓       ['1000', '2000', '3000', '4000', '42']
  scale 0.0 (off)      seperate_all_ignorant      5      ✓       ['1000', '2000', '3000', '4000', '42']
  scale 0.0 (off)      seperate_mixed_5050        5      ✓       ['1000', '2000', '3000', '4000', '42']
  scale 0.0 (off)      cluster_aware_ignorant     5      ✓       ['1000', '2000', '3000', '4000', '42']
  scale 0.2            seperate_all_aware         5      ✓       ['scale_0.2_1000', 'scale_0.2_2000', 'scale_0.2_3000', 'scale_0.2_4000', 'scale_0.2_42']
  scale 0.2            seperate_all_ignorant      5      ✓       ['scale_0.2_1000', 'scale_0.2_2000', 'scale_0.2_3000', 'scale_0.2_4000', 'scale_0.2_42']
  scale 0.2            seperate_mixed_5050        5      ✓       ['scale_0.2_1000', 'scale_0.2_2000', 'scale_0.2_3000', 'scale_0.2_4000', 'scale_0.2_42']
  scale 0.2            cluster_aware_ignorant     5

In [10]:
# ----------------------- ablation: long view (mean ± sd) -----------------------
# Ablation has one extra column vs. the main table: the average LoRA scale
# (`dynamic_scale`) actually applied per step, averaged over the test episodes.
# For fixed-scale rows this equals the fixed value; for adaptive_gt it reflects
# the time-averaged interpolated scale.
ABL_METRICS = METRICS + [('avg_lora_scale', 'LoRA scale', False, 3)]

abl_rows = []
for label, _, _ in ABL_METHODS:
    for sc in SCENARIOS:
        seed_dicts = abl_loaded[label][sc]
        row = {'Row': label, 'Scenario': sc}
        for mkey, mlabel, in_pct, dec in ABL_METRICS:
            vals = [s.get(mkey) for s in seed_dicts.values() if s.get(mkey) is not None]
            if vals:
                m = float(np.mean(vals))
                sd = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
                row[mlabel] = fmt(m, sd, in_pct, dec)
            else:
                row[mlabel] = '—'
        abl_rows.append(row)

df_ablation = pd.DataFrame(abl_rows)
df_ablation

,Row,Scenario,SR (%),CR (%),NT (s),PL (m),ITR (%),SocD (m),LoRA scale
0,scale 0.0 (off),seperate_all_aware,100.00 ± 0.00,0.00 ± 0.00,16.24 ± 0.47,23.49 ± 0.64,2.61 ± 0.32,0.439 ± 0.011,0.000 ± 0.000
1,scale 0.0 (off),seperate_all_ignorant,97.20 ± 0.40,2.80 ± 0.40,16.59 ± 0.47,23.53 ± 0.62,2.30 ± 0.29,0.451 ± 0.004,0.000 ± 0.000
2,scale 0.0 (off),seperate_mixed_5050,98.56 ± 0.73,1.36 ± 0.73,16.61 ± 0.58,23.79 ± 0.74,2.62 ± 0.23,0.439 ± 0.010,0.000 ± 0.000
3,scale 0.0 (off),cluster_aware_ignorant,97.12 ± 0.77,2.88 ± 0.77,15.47 ± 0.68,22.40 ± 0.89,3.89 ± 0.40,0.431 ± 0.002,0.000 ± 0.000
4,scale 0.2,seperate_all_aware,99.92 ± 0.18,0.08 ± 0.18,14.34 ± 0.43,21.56 ± 0.52,3.58 ± 0.58,0.441 ± 0.016,0.200 ± 0.000
5,scale 0.2,seperate_all_ignorant,93.84 ± 0.78,6.16 ± 0.78,15.01 ± 0.34,21.82 ± 0.46,3.66 ± 0.29,0.439 ± 0.007,0.200 ± 0.000
6,scale 0.2,seperate_mixed_5050,98.00 ± 0.00,2.00 ± 0.00,14.88 ± 0.28,22.03 ± 0.45,3.68 ± 0.29,0.436 ± 0.005,0.200 ± 0.000
7,scale 0.2,cluster_aware_ignorant,96.00 ± 0.40,4.00 ± 0.40,13.76 ± 0.40,20.73 ± 0.63,4.94 ± 0.37,0.434 ± 0.007,0.200 ± 0.000
8,scale 0.4,seperate_all_aware,99.92 ± 0.18,0.08 ± 0.18,12.61 ± 0.16,19.84 ± 0.31,5.01 ± 0.72,0.437 ± 0.009,0.400 ± 0.000
9,scale 0.4,seperate_all_ignorant,88.16 ± 2.49,11.84 ± 2.49,13.90 ± 0.28,20.47 ± 0.50,5.85 ± 0.49,0.429 ± 0.007,0.400 ± 0.000


In [11]:
# ----------------------- ablation: wide pivot (rows=Row, cols=Scenario × Metric) -----------------------
abl_long = df_ablation.melt(id_vars=['Row', 'Scenario'], var_name='Metric', value_name='Value')
abl_wide = abl_long.pivot_table(index='Row', columns=['Scenario', 'Metric'], values='Value', aggfunc='first')
abl_wide = abl_wide.reindex([m[0] for m in ABL_METHODS])
metric_order = [m[1] for m in ABL_METRICS]
abl_wide = abl_wide.reindex(columns=pd.MultiIndex.from_product([SCENARIOS, metric_order], names=['Scenario', 'Metric']))

df_ablation.to_csv('giant_table_ablation_long.csv', index=False)
abl_wide.to_csv('giant_table_ablation_wide.csv')
print('Wrote: giant_table_ablation_long.csv, giant_table_ablation_wide.csv')
abl_wide

Wrote: giant_table_ablation_long.csv, giant_table_ablation_wide.csv


Scenario        seperate_all_aware                                                                                     seperate_all_ignorant                                                          \
Metric                      SR (%)       CR (%)        NT (s)        PL (m)      ITR (%)       SocD (m)     LoRA scale                SR (%)        CR (%)        NT (s)        PL (m)       ITR (%)   
Row                                                                                                                                                                                                    
scale 0.0 (off)      100.00 ± 0.00  0.00 ± 0.00  16.24 ± 0.47  23.49 ± 0.64  2.61 ± 0.32  0.439 ± 0.011  0.000 ± 0.000          97.20 ± 0.40   2.80 ± 0.40  16.59 ± 0.47  23.53 ± 0.62   2.30 ± 0.29   
scale 0.2             99.92 ± 0.18  0.08 ± 0.18  14.34 ± 0.43  21.56 ± 0.52  3.58 ± 0.58  0.441 ± 0.016  0.200 ± 0.000          93.84 ± 0.78   6.16 ± 0.78  15.01 ± 0.34  21.82 ± 0.46   3.66 ± 0.29   
scale 0.4             99.92 ± 0.18  0.08 ± 0.18  12.61 ± 0.16  19.84 ± 0.31  5.01 ± 0.72  0.437 ± 0.009  0.400 ± 0.000          88.16 ± 2.49  11.84 ± 2.49  13.90 ± 0.28  20.47 ± 0.50   5.85 ± 0.49   
scale 0.6             99.68 ± 0.33  0.32 ± 0.33  11.73 ± 0.27  18.96 ± 0.42  6.03 ± 0.59  0.425 ± 0.011  0.600 ± 0.000          83.12 ± 1.78  16.88 ± 1.78  13.25 ± 0.25  19.62 ± 0.50   7.67 ± 0.38   
scale 0.8             99.52 ± 0.52  0.48 ± 0.52  11.16 ± 0.13  18.37 ± 0.27  6.68 ± 0.72  0.424 ± 0.014  0.800 ± 0.000          77.36 ± 2.92  22.64 ± 2.92  12.66 ± 0.26  18.72 ± 0.35   9.11 ± 0.40   
scale 1.0             99.28 ± 0.33  0.72 ± 0.33  10.94 ± 0.11  18.16 ± 0.28  7.33 ± 0.74  0.417 ± 0.011  1.000 ± 0.000          73.12 ± 3.99  26.88 ± 3.99  12.35 ± 0.25  18.14 ± 0.47  10.31 ± 0.46   
adaptive_gt           99.28 ± 0.33  0.72 ± 0.33  10.94 ± 0.11  18.16 ± 0.28  7.33 ± 0.74  0.417 ± 0.011  1.000 ± 0.000          97.04 ± 0.36   2.96 ± 0.36  16.51 ± 0.44  23.44 ± 0.59   2.39 ± 0.25   

Scenario                                      seperate_mixed_5050                                                                                      cluster_aware_ignorant                \
Metric                SocD (m)     LoRA scale              SR (%)        CR (%)        NT (s)        PL (m)      ITR (%)       SocD (m)     LoRA scale                 SR (%)        CR (%)   
Row                                                                                                                                                                                           
scale 0.0 (off)  0.451 ± 0.004  0.000 ± 0.000        98.56 ± 0.73   1.36 ± 0.73  16.61 ± 0.58  23.79 ± 0.74  2.62 ± 0.23  0.439 ± 0.010  0.000 ± 0.000           97.12 ± 0.77   2.88 ± 0.77   
scale 0.2        0.439 ± 0.007  0.200 ± 0.000        98.00 ± 0.00   2.00 ± 0.00  14.88 ± 0.28  22.03 ± 0.45  3.68 ± 0.29  0.436 ± 0.005  0.200 ± 0.000           96.00 ± 0.40   4.00 ± 0.40   
scale 0.4        0.429 ± 0.007  0.400 ± 0.000        96.24 ± 1.34   3.76 ± 1.34  13.65 ± 0.40  20.66 ± 0.56  5.49 ± 0.51  0.430 ± 0.006  0.400 ± 0.000           94.64 ± 0.83   5.36 ± 0.83   
scale 0.6        0.421 ± 0.008  0.600 ± 0.000        94.16 ± 1.15   5.84 ± 1.15  12.66 ± 0.27  19.61 ± 0.42  7.01 ± 0.32  0.422 ± 0.009  0.600 ± 0.000           92.32 ± 1.73   7.68 ± 1.73   
scale 0.8        0.414 ± 0.004  0.800 ± 0.000        89.60 ± 1.13  10.40 ± 1.13  12.19 ± 0.15  18.90 ± 0.22  8.13 ± 0.24  0.424 ± 0.008  0.800 ± 0.000           90.64 ± 1.08   9.36 ± 1.08   
scale 1.0        0.410 ± 0.005  1.000 ± 0.000        86.08 ± 2.79  13.92 ± 2.79  11.85 ± 0.18  18.44 ± 0.33  9.31 ± 0.60  0.411 ± 0.011  1.000 ± 0.000           88.80 ± 1.85  11.20 ± 1.85   
adaptive_gt      0.448 ± 0.005  0.017 ± 0.006        95.52 ± 1.53   4.48 ± 1.53  13.63 ± 0.40  20.64 ± 0.53  5.87 ± 0.39  0.431 ± 0.006  0.502 ± 0.005           95.36 ± 0.92   4.64 ± 0.92   

Scenario                                                                                

# Rebuttal

Action-space interpolation baseline for the rebuttal. Run this sweep first so `trained_models/LoraF_invi_visi_rank_1/test/all_evaluations.json` contains `adaptive_action_gt` entries for all scenarios and seeds.

```bash
GPUS=(0 1 2 3)
MAX_PARALLEL=4
JOB_INDEX=0
RUNNING=0

for SEED in 42 1000 2000 3000 4000; do
  for SCENARIO in seperate_all_aware seperate_all_ignorant seperate_mixed_5050 cluster_aware_ignorant; do
    GPU=${GPUS[$((JOB_INDEX % ${#GPUS[@]}))]}
    CUDA_VISIBLE_DEVICES=$GPU python3 test.py \
      --model_dir trained_models/LoraF_invi_visi_rank_1 \
      --test_model 03400.pt \
      --adaptive_lora_scenario "$SCENARIO" \
      --lora_behaviour adaptive_action_gt \
      --human_num 20 \
      --test_size 250 \
      --awareness_eval off \
      --save_episode_dump never \
      --seed "$SEED" \
      --exp_id "$SEED" &

    JOB_INDEX=$((JOB_INDEX + 1))
    RUNNING=$((RUNNING + 1))
    if (( RUNNING >= MAX_PARALLEL )); then
      wait -n
      RUNNING=$((RUNNING - 1))
    fi
  done
done
wait
```

In [ ]:
# ----------------------- Rebuttal: adaptive_action_gt -----------------------
import os, json
import numpy as np
import pandas as pd

REBUTTAL_MODEL_DIR = 'trained_models/LoraF_invi_visi_rank_1'
REBUTTAL_BEHAVIOUR = 'adaptive_action_gt'
REBUTTAL_SEEDS = ['42', '1000', '2000', '3000', '4000']
REBUTTAL_SCENARIOS = [
    'seperate_all_aware',
    'seperate_all_ignorant',
    'seperate_mixed_5050',
    'cluster_aware_ignorant',
]
REBUTTAL_METRICS = [
    ('success_rate', 'SR (%)', True, 2),
    ('collision_rate', 'CR (%)', True, 2),
    ('avg_nav_time', 'NT (s)', False, 2),
    ('avg_path_length', 'PL (m)', False, 2),
    ('avg_intrusion_ratio_pct', 'ITR (%)', False, 2),
    ('avg_min_social_distance', 'SocD (m)', False, 3),
    ('avg_lora_scale', 'Avg kappa', False, 3),
]

def _fmt_rebuttal(vals, in_percent, decimals):
    if not vals:
        return '—'
    arr = np.asarray(vals, dtype=float)
    mean = float(arr.mean())
    sd = float(arr.std(ddof=1)) if len(arr) > 1 else 0.0
    if in_percent:
        mean *= 100
        sd *= 100
    return f'{mean:.{decimals}f} ± {sd:.{decimals}f}'

agg_path = os.path.join(REBUTTAL_MODEL_DIR, 'test', 'all_evaluations.json')
if not os.path.exists(agg_path):
    raise FileNotFoundError(f'Missing aggregate file: {agg_path}')

with open(agg_path) as f:
    aggregate = json.load(f)

coverage_rows = []
table_rows = []
for scenario in REBUTTAL_SCENARIOS:
    summaries = {}
    for seed in REBUTTAL_SEEDS:
        key = f'{scenario}_{REBUTTAL_BEHAVIOUR}_exp{seed}'
        ent = aggregate.get(key)
        if ent is not None:
            summaries[seed] = ent.get('summary', {})

    found = sorted(summaries.keys(), key=lambda x: REBUTTAL_SEEDS.index(x))
    coverage_rows.append({
        'Scenario': scenario,
        'Expected seeds': len(REBUTTAL_SEEDS),
        'Found seeds': len(found),
        'Complete': len(found) == len(REBUTTAL_SEEDS),
        'Seeds': found,
    })

    row = {'Method': 'Action-space adaptive GT', 'Scenario': scenario}
    for metric_key, label, in_percent, decimals in REBUTTAL_METRICS:
        vals = [s.get(metric_key) for s in summaries.values() if s.get(metric_key) is not None]
        row[label] = _fmt_rebuttal(vals, in_percent, decimals)
    table_rows.append(row)

rebuttal_coverage = pd.DataFrame(coverage_rows)
display(rebuttal_coverage)

df_rebuttal_adaptive_action_gt = pd.DataFrame(table_rows)
df_rebuttal_adaptive_action_gt.to_csv('giant_table_rebuttal_adaptive_action_gt.csv', index=False)
df_rebuttal_adaptive_action_gt


## Notes

- **Aggressive row caveat:** "GenSafeNav (aggressive)" is currently sourced from `trained_models/FullFineTune_invi_visi` (`always_off`) rather than a true `always_on` LoRA run. FullFineTune is a different method (no LoRA gating — the entire policy was retrained on invi→visi), so treat this row as the *upper-bound aggressive baseline* rather than literally "GenSafeNav with the LoRA branch always on". To get a true LoRA `always_on` row later, run:
  ```bash
  for s in 42 1000 2000 3000 4000; do
    for sc in seperate_all_aware seperate_all_ignorant seperate_mixed_5050 cluster_aware_ignorant; do
      python test.py --model_dir trained_models/LoraF_invi_visi_rank_1 --test_model 03400.pt \
        --lora_behaviour always_on --adaptive_lora_scenario $sc \
        --seed $s --exp_id ${sc}_always_on_exp${s}
    done
  done
  ```
  then point the aggressive row in cell 1 back to `LoraF_invi_visi_rank_1` with behaviour `always_on`.

- **Baseline SD = 0:** SF / ORCA / CrowdNav++ rows show `± 0.0` for every metric because their `all_evaluations.json` entries are bit-identical across exp_ids 42/1000/2000/3000/4000 — those test runs were never reseeded (same simulator state, just relabelled). Re-test them with `--seed $SEED` actually wired in (the way `run_all_results.sh` does for LoraF) and the ± numbers will populate.